In [1]:
import pandas as pd
from pathlib import Path
raw_dir = Path("../data/raw")
files = sorted(raw_dir.glob("fuelwatch_2026_*.csv"))
files

[WindowsPath('../data/raw/fuelwatch_2026_04.csv'),
 WindowsPath('../data/raw/fuelwatch_2026_05.csv'),
 WindowsPath('../data/raw/fuelwatch_2026_06.csv')]

## 2. Inspect a sample monthly dataset

Before combining all monthly files, one CSV file is loaded to inspect its dimensions, column names, and sample records.

This confirms that the dataset has been downloaded correctly and that its structure is suitable for further processing.

In [2]:
df = pd.read_csv(files[0])

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (85007, 10)

Columns:
['PUBLISH_DATE', 'TRADING_NAME', 'BRAND_DESCRIPTION', 'PRODUCT_DESCRIPTION', 'PRODUCT_PRICE', 'ADDRESS', 'LOCATION', 'POSTCODE', 'AREA_DESCRIPTION', 'REGION_DESCRIPTION']


,PUBLISH_DATE,TRADING_NAME,BRAND_DESCRIPTION,PRODUCT_DESCRIPTION,PRODUCT_PRICE,ADDRESS,LOCATION,POSTCODE,AREA_DESCRIPTION,REGION_DESCRIPTION
0,01/04/2026,53 Mile Roadhouse,United,ULP,260.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel
1,01/04/2026,53 Mile Roadhouse,United,Diesel,329.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel
2,01/04/2026,53 Mile Roadhouse,United,98 RON,287.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel
3,01/04/2026,7-Eleven Alkimos,7-Eleven,ULP,255.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro
4,01/04/2026,7-Eleven Alkimos,7-Eleven,PULP,270.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro


## 3. Review data types and completeness

The `info()` method provides a structural summary of the dataset, including:

- total number of rows;
- column names;
- number of non-null values;
- data type of each column;
- approximate memory usage.

This helps identify columns that require type conversion, such as converting the publication date from text to a datetime format.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85007 entries, 0 to 85006
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PUBLISH_DATE         85007 non-null  object 
 1   TRADING_NAME         85007 non-null  object 
 2   BRAND_DESCRIPTION    85007 non-null  object 
 3   PRODUCT_DESCRIPTION  85007 non-null  object 
 4   PRODUCT_PRICE        85007 non-null  float64
 5   ADDRESS              85007 non-null  object 
 6   LOCATION             85007 non-null  object 
 7   POSTCODE             85007 non-null  int64  
 8   AREA_DESCRIPTION     85007 non-null  object 
 9   REGION_DESCRIPTION   85007 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 6.5+ MB


## 4. Check missing values and duplicate records

This section evaluates basic data quality by checking:

- the number of missing values in each column;
- the number of completely duplicated rows.

Missing or duplicated records can affect summary statistics and model results, so they should be identified before combining and cleaning the datasets.

In [4]:
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

PUBLISH_DATE           0
TRADING_NAME           0
BRAND_DESCRIPTION      0
PRODUCT_DESCRIPTION    0
PRODUCT_PRICE          0
ADDRESS                0
LOCATION               0
POSTCODE               0
AREA_DESCRIPTION       0
REGION_DESCRIPTION     0
dtype: int64

Duplicate rows: 0


## 5. Combine monthly FuelWatch datasets

All monthly CSV files are loaded and combined into a single DataFrame.

A `SOURCE_FILE` column is added before concatenation so that each row can be traced back to its original monthly file. This is useful for validation and troubleshooting.

In [5]:
dataframes = []

for file in files:
    monthly_df = pd.read_csv(file)
    monthly_df["SOURCE_FILE"] = file.name
    dataframes.append(monthly_df)

fuel_df = pd.concat(dataframes, ignore_index=True)

print("Combined shape:", fuel_df.shape)
fuel_df.head()

Combined shape: (289658, 11)


,PUBLISH_DATE,TRADING_NAME,BRAND_DESCRIPTION,PRODUCT_DESCRIPTION,PRODUCT_PRICE,ADDRESS,LOCATION,POSTCODE,AREA_DESCRIPTION,REGION_DESCRIPTION,SOURCE_FILE
0,01/04/2026,53 Mile Roadhouse,United,ULP,260.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,fuelwatch_2026_04.csv
1,01/04/2026,53 Mile Roadhouse,United,Diesel,329.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,fuelwatch_2026_04.csv
2,01/04/2026,53 Mile Roadhouse,United,98 RON,287.9,31 South Western Hwy,PINJARRA,6208,Murray,Peel,fuelwatch_2026_04.csv
3,01/04/2026,7-Eleven Alkimos,7-Eleven,ULP,255.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro,fuelwatch_2026_04.csv
4,01/04/2026,7-Eleven Alkimos,7-Eleven,PULP,270.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro,fuelwatch_2026_04.csv


## 6. Standardise column names

Column names are converted to lowercase and surrounding whitespace is removed.

Using consistent lowercase column names makes the code easier to read and reduces the risk of errors caused by inconsistent capitalisation.

In [6]:
fuel_df.columns = (
    fuel_df.columns
    .str.strip()
    .str.lower()
)

fuel_df.columns.tolist()

['publish_date',
 'trading_name',
 'brand_description',
 'product_description',
 'product_price',
 'address',
 'location',
 'postcode',
 'area_description',
 'region_description',
 'source_file']

## 7. Convert publication dates to datetime

The `publish_date` column is converted from text into Pandas datetime format.

The source files use the Australian date format `day/month/year`. Converting this column enables time-based analysis such as grouping prices by month, weekday, and date range.

In [7]:
fuel_df["publish_date"] = pd.to_datetime(
    fuel_df["publish_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

fuel_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289658 entries, 0 to 289657
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   publish_date         289658 non-null  datetime64[ns]
 1   trading_name         289658 non-null  object        
 2   brand_description    289658 non-null  object        
 3   product_description  289658 non-null  object        
 4   product_price        289658 non-null  float64       
 5   address              289658 non-null  object        
 6   location             289658 non-null  object        
 7   postcode             289658 non-null  int64         
 8   area_description     289658 non-null  object        
 9   region_description   289658 non-null  object        
 10  source_file          289658 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(8)
memory usage: 24.3+ MB


## 8. Validate the dataset date range

This section checks the earliest and latest dates in the combined dataset and counts the number of observations for each month.

The results confirm whether all expected monthly files were loaded correctly.

In [8]:
print("Earliest date:", fuel_df["publish_date"].min())
print("Latest date:", fuel_df["publish_date"].max())

print(
    fuel_df.groupby(
        fuel_df["publish_date"].dt.to_period("M")
    ).size()
)

Earliest date: 2026-04-01 00:00:00
Latest date: 2026-06-30 00:00:00
publish_date
2026-04     85007
2026-05    103501
2026-06    101150
Freq: M, dtype: int64


## 9. Explore categorical variables

The unique fuel products and geographic regions are examined using frequency counts.

This helps determine which fuel types and regions should be included in the initial analysis. The first version of the project will primarily focus on ULP prices in the Perth metropolitan area.

In [9]:
print("Fuel types:")
print(
    fuel_df["product_description"]
    .value_counts()
)

print("\nRegions:")
print(
    fuel_df["region_description"]
    .value_counts()
)

Fuel types:
product_description
ULP             78609
98 RON          63579
Diesel          54678
PULP            47013
Brand Diesel    41055
LPG              3541
E85              1183
Name: count, dtype: int64

Regions:
region_description
Metro                   167360
South-West               28580
Wheatbelt                20876
Peel                     13709
Mid-West                 13027
Great Southern           12401
Goldfields-Esperance     12216
Pilbara                   8617
Kimberley                 8068
Gascoyne                  4804
Name: count, dtype: int64


## 10. Recheck data quality after concatenation

After combining all monthly files, missing values and duplicated rows are checked again.

This ensures that the concatenation process did not introduce unexpected data quality issues.

In [10]:
print("Missing values:")
print(fuel_df.isna().sum())

print("\nDuplicate rows:", fuel_df.duplicated().sum())

Missing values:
publish_date           0
trading_name           0
brand_description      0
product_description    0
product_price          0
address                0
location               0
postcode               0
area_description       0
region_description     0
source_file            0
dtype: int64

Duplicate rows: 0


# Data Cleaning

## Rename columns

The original FuelWatch column names are renamed to shorter and more descriptive names for easier analysis.

In [11]:
fuel_df = fuel_df.rename(
    columns={
        "publish_date": "date",
        "trading_name": "station_name",
        "brand_description": "brand",
        "product_description": "fuel_type",
        "product_price": "price_cpl",
        "location": "suburb",
        "area_description": "area",
        "region_description": "region",
        "source_file": "source_file"
    }
)

fuel_df.columns.tolist()

['date',
 'station_name',
 'brand',
 'fuel_type',
 'price_cpl',
 'address',
 'suburb',
 'postcode',
 'area',
 'region',
 'source_file']

## Clean text columns

Leading and trailing whitespace is removed from text columns to ensure consistent category names.

In [12]:
text_columns = [
    "station_name",
    "brand",
    "fuel_type",
    "address",
    "suburb",
    "area",
    "region"
]

for column in text_columns:
    fuel_df[column] = fuel_df[column].str.strip()

## Validate fuel prices

Summary statistics are reviewed to identify potentially invalid or extreme fuel prices.

In [13]:
fuel_df["price_cpl"].describe()
fuel_df.sort_values("price_cpl").head(10)
fuel_df.sort_values("price_cpl", ascending=False).head(10)

,date,station_name,brand,fuel_type,price_cpl,address,suburb,postcode,area,region,source_file
218805,2026-06-09,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
282898,2026-06-28,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
269398,2026-06-24,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
205332,2026-06-05,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
154901,2026-05-21,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_05.csv
161602,2026-05-23,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_05.csv
175028,2026-05-27,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_05.csv
201964,2026-06-04,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
262646,2026-06-22,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv
266022,2026-06-23,Yakanarra Community Store,Independent,Diesel,450.0,Nyinyijarta Rd,ST GEORGE RANGES,6728,St George Ranges,Kimberley,fuelwatch_2026_06.csv


## Create station identifiers

A station identifier is created by combining the station name, address, and suburb. This allows prices from the same station to be tracked across multiple dates.

In [16]:
fuel_df["station_id"] = (
    fuel_df["station_name"].str.lower()
    + "_"
    + fuel_df["address"].str.lower()
    + "_"
    + fuel_df["suburb"].str.lower()
)

# Validate
fuel_df[
    ["station_id", "station_name", "address", "suburb"]
].head()

,station_id,station_name,address,suburb
0,53 mile roadhouse_31 south western hwy_pinjarra,53 Mile Roadhouse,31 South Western Hwy,PINJARRA
1,53 mile roadhouse_31 south western hwy_pinjarra,53 Mile Roadhouse,31 South Western Hwy,PINJARRA
2,53 mile roadhouse_31 south western hwy_pinjarra,53 Mile Roadhouse,31 South Western Hwy,PINJARRA
3,7-eleven alkimos_10 carlsbad prom_alkimos,7-Eleven Alkimos,10 Carlsbad Prom,ALKIMOS
4,7-eleven alkimos_10 carlsbad prom_alkimos,7-Eleven Alkimos,10 Carlsbad Prom,ALKIMOS


## Create time-based features

Additional date features are created for analysing fuel prices by year, month, weekday, and weekend status.

In [ ]:
fuel_df["year"] = fuel_df["date"].dt.year
fuel_df["month"] = fuel_df["date"].dt.month
fuel_df["month_name"] = fuel_df["date"].dt.month_name()
fuel_df["day_of_week"] = fuel_df["date"].dt.day_name()
fuel_df["day_number"] = fuel_df["date"].dt.dayofweek
fuel_df["is_weekend"] = fuel_df["day_number"] >= 5

,date,year,month,month_name,day_of_week,is_weekend
0,2026-04-01,2026,4,April,Wednesday,False
1,2026-04-01,2026,4,April,Wednesday,False
2,2026-04-01,2026,4,April,Wednesday,False
3,2026-04-01,2026,4,April,Wednesday,False
4,2026-04-01,2026,4,April,Wednesday,False


## Check business-key duplicates

Duplicate records are checked using date, station, and fuel type as the business key.

In [19]:
business_key = [
    "date",
    "station_id",
    "fuel_type"
]

business_duplicates = fuel_df.duplicated(
    subset=business_key,
    keep=False
)

print(
    "Business-key duplicate rows:",
    business_duplicates.sum()
)

Business-key duplicate rows: 0


## Create the initial Perth ULP analysis dataset

The first analysis dataset focuses on regular unleaded petrol in the Perth metropolitan region.

In [20]:
perth_ulp_df = fuel_df[
    (fuel_df["fuel_type"] == "ULP")
    & (fuel_df["region"] == "Metro")
].copy()

print("Perth ULP shape:", perth_ulp_df.shape)

perth_ulp_df.head()

Perth ULP shape: (40700, 18)


,date,station_name,brand,fuel_type,price_cpl,address,suburb,postcode,area,region,source_file,station_id,year,month,month_name,day_of_week,day_number,is_weekend
3,2026-04-01,7-Eleven Alkimos,7-Eleven,ULP,255.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro,fuelwatch_2026_04.csv,7-eleven alkimos_10 carlsbad prom_alkimos,2026,4,April,Wednesday,2,False
7,2026-04-01,7-Eleven Ascot,7-Eleven,ULP,255.9,194 Great Eastern Hwy,ASCOT,6104,South of River,Metro,fuelwatch_2026_04.csv,7-eleven ascot_194 great eastern hwy_ascot,2026,4,April,Wednesday,2,False
11,2026-04-01,7-Eleven Balcatta,7-Eleven,ULP,255.9,174 Balcatta Rd,BALCATTA,6021,North of River,Metro,fuelwatch_2026_04.csv,7-eleven balcatta_174 balcatta rd_balcatta,2026,4,April,Wednesday,2,False
15,2026-04-01,7-Eleven Baldivis,7-Eleven,ULP,255.9,370 Baldivis Rd,BALDIVIS,6171,South of River,Metro,fuelwatch_2026_04.csv,7-eleven baldivis_370 baldivis rd_baldivis,2026,4,April,Wednesday,2,False
19,2026-04-01,7-Eleven Balga,7-Eleven,ULP,255.9,102 Princess Rd,BALGA,6061,North of River,Metro,fuelwatch_2026_04.csv,7-eleven balga_102 princess rd_balga,2026,4,April,Wednesday,2,False


## Validate the cleaned dataset

The cleaned dataset is checked for missing values, duplicates, date coverage, and basic fuel price statistics.

In [21]:
print("Shape:", perth_ulp_df.shape)

print("\nDate range:")
print(perth_ulp_df["date"].min())
print(perth_ulp_df["date"].max())

print("\nMissing values:")
print(perth_ulp_df.isna().sum())

print("\nPrice summary:")
print(perth_ulp_df["price_cpl"].describe())

print(
    "\nUnique stations:",
    perth_ulp_df["station_id"].nunique()
)

Shape: (40700, 18)

Date range:
2026-04-01 00:00:00
2026-06-30 00:00:00

Missing values:
date            0
station_name    0
brand           0
fuel_type       0
price_cpl       0
address         0
suburb          0
postcode        0
area            0
region          0
source_file     0
station_id      0
year            0
month           0
month_name      0
day_of_week     0
day_number      0
is_weekend      0
dtype: int64

Price summary:
count    40700.000000
mean       182.070907
std         22.858751
min        136.500000
25%        165.900000
50%        177.900000
75%        193.900000
max        269.900000
Name: price_cpl, dtype: float64

Unique stations: 451


## Export cleaned data

The cleaned Perth ULP dataset is exported to the processed data directory for use in exploratory analysis and dashboard development.

In [22]:
processed_path = Path(
    "../data/processed/perth_ulp_2026_04_to_06.csv"
)

perth_ulp_df.to_csv(
    processed_path,
    index=False
)

print("Saved to:", processed_path)

Saved to: ..\data\processed\perth_ulp_2026_04_to_06.csv


In [23]:
check_df = pd.read_csv(
    "../data/processed/perth_ulp_2026_04_to_06.csv"
)

print("Saved shape:", check_df.shape)
check_df.head()

Saved shape: (40700, 18)


,date,station_name,brand,fuel_type,price_cpl,address,suburb,postcode,area,region,source_file,station_id,year,month,month_name,day_of_week,day_number,is_weekend
0,2026-04-01,7-Eleven Alkimos,7-Eleven,ULP,255.9,10 Carlsbad Prom,ALKIMOS,6038,North of River,Metro,fuelwatch_2026_04.csv,7-eleven alkimos_10 carlsbad prom_alkimos,2026,4,April,Wednesday,2,False
1,2026-04-01,7-Eleven Ascot,7-Eleven,ULP,255.9,194 Great Eastern Hwy,ASCOT,6104,South of River,Metro,fuelwatch_2026_04.csv,7-eleven ascot_194 great eastern hwy_ascot,2026,4,April,Wednesday,2,False
2,2026-04-01,7-Eleven Balcatta,7-Eleven,ULP,255.9,174 Balcatta Rd,BALCATTA,6021,North of River,Metro,fuelwatch_2026_04.csv,7-eleven balcatta_174 balcatta rd_balcatta,2026,4,April,Wednesday,2,False
3,2026-04-01,7-Eleven Baldivis,7-Eleven,ULP,255.9,370 Baldivis Rd,BALDIVIS,6171,South of River,Metro,fuelwatch_2026_04.csv,7-eleven baldivis_370 baldivis rd_baldivis,2026,4,April,Wednesday,2,False
4,2026-04-01,7-Eleven Balga,7-Eleven,ULP,255.9,102 Princess Rd,BALGA,6061,North of River,Metro,fuelwatch_2026_04.csv,7-eleven balga_102 princess rd_balga,2026,4,April,Wednesday,2,False
